# Six-model B1 logistic stack

**Later illustrated companion; not the executed training notebook.** Historical metrics below are quoted from copied run records, never represented as notebook outputs.

The stack was explored after inspecting other dev results, so comparisons across stack families on the same official dev set are post-selection and may be optimistic. The train-calibration CV is not an unseen test estimate. Reproduction needs the saved per-model probability arrays and selected checkpoint lineage in the original local B1 archive; the small package includes the builder script and full hash/provenance record, but not all probabilities or model weights.

## Task and architecture

This is a prediction-level ensemble, not a newly trained Phase 1/B1 backbone. It joins saved calibration, dev and test probabilities from: v7_transformer, v6_transformer, v5_gru, v4_reference, v7_gru, v6_gru. An L2-regularized logistic regression learns the mixing weights from 512 training-calibration examples; each original model retains its own checkpoint. Decision threshold 0.500000.

**Adaptation:** Six B1 inputs, exploratory model family; L2 logistic stack C=1.0 fit on 512 disjoint train calibration examples; five-fold calibration CV; threshold 0.5. Official dev inspected for candidate comparison, so dev estimate is post-selection optimistic; hidden score unknown.

Data path: official train → training-only fit/selection → official labeled dev report → 1,517-label-free test predictions. The original `code/` and `evidence/` directories contain the audit trail.

## Recorded result

Five-fold train-calibration CV macro-F1 **0.704825** on 512 cases. Exploratory official 994-case dev macro-F1 **0.703217**. Verified 1,517-ID ZIP SHA-256 `515dec48d787b54381b4463d81c5d498d6908dc47aca268b345c7427ec96303d`.

Local official dev, calibration CV, and Codabench scores are different measurements. A historical score is not an output of this notebook.

## Source excerpt: `build_b1_stack6_submission.py`

The complete source is in `code/`. This excerpt illustrates the exact preserved implementation; it is not a training replay.

In [ ]:
import datetime
import hashlib
import json
import pathlib
import sys
import zipfile

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

root = pathlib.Path(sys.argv[1])
ready = pathlib.Path(sys.argv[2])
outputs = root / 'outputs'
models = {
    'v7_transformer': ('v7', 'B1_Window_TransformerOnly'),
    'v6_transformer': ('v6', 'B1_Window_TransformerOnly'),
    'v5_gru': ('v5', 'B1_Window_GRUOnly'),
    'v4_reference': ('v4', 'B1_FullData_Reference'),
    'v7_gru': ('v7', 'B1_Window_GRUOnly'),
    'v6_gru': ('v6', 'B1_Window_GRUOnly'),
}
assert (root / 'VERIFIED_HEADS_V7_SEGMENT3_BACKUPS.json').exists()
assert (root / 'VERIFIED_HEADS_V7_SEGMENT3_EXPORTS.json').exists()


def sha256(path):
    with path.open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()


def load_model(version, name):
    base = outputs / version / 'CS781_B1_Hierarchy_Depth' / 'runs' / name / 'exports'
    dirs = [p for p in base.iterdir() if p.is_dir() and p.name.startswith('segment_')]
    exp = max(dirs, key=lambda p: int(p.name.split('_')[-1]))
    data = {}
    hashes = {}
    for split in ('calibration', 'dev', 'test'):
        path = exp / f'{split}_probabilities.npz'
        hashes[split] = {'path': str(path), 'sha256': sha256(path)}
        with np.load(path, allow_pickle=False) as arr:
            ids = arr['ids'].tolist()
            logits = arr['logits']
            assert logits.shape == (len(ids), 2)
            assert len(ids) == len(set(ids))
            margin = (logits[:, 1] - logits[:, 0]).astype(np.float64)
            assert np.isfinite(margin).all()
            entry = {'ids': ids, 'margin': margin}
            if split == 'calibration':
                entry['labels'] = arr['labels'].astype(int)
            elif split == 'dev':
                rows = json.loads((exp / 'dev_truth.json').read_text(encoding='utf-8'))
                gold = {r['id']: int(r['gold']) for r in rows}
                assert len(gold) == len(ids)
                entry['labels'] = np.array([gold[i] for i in ids], dtype=int)
            data[split] = entry
    return exp, data, hashes

loaded = {key: load_model(*spec) for key, spec in models.items()}
order = list(models)
base = loaded[order[0]][1]
with zipfile.ZipFile(ready / 'B1_Transformer_v7.zip') as archive:
    official_ids = [r['id'] for r in json.loads(archive.read('prediction.json'))]
assert len(official_ids) == len(set(official_ids)) == 1517


def matrix(split):
    ids = base[split]['ids']
    assert len(ids) == (1517 if split == 'test' else 512 if split == 'calibration' else 994)
    columns = []
    labels = base[split].get('labels')
    for key in order:
        entry = loaded[key][1][split]
        positions = {identifier: i for i, identifier in enumerate(entry['ids'])}
        assert set(positions) == set(ids)
        idx = [positions[identifier] for identifier in ids]
        columns.append(entry['margin'][idx])
        if labels is not None:
            assert np.array_equal(entry['labels'][idx], labels)
    return ids, np.column_stack(columns), labels

cal_ids, x_cal, y_cal = matrix('calibration')

## Reproduction boundary

The package has code, configs, selected records, and prediction ZIP. Original model weights and official data are retained in the local archives, not duplicated here. Paths in the original configs must be remapped before an authorized rerun.